In [10]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = Path(r"C:\Users\maria\Desktop\EEG-special-course")
PROCESSED_ROOT = PROJECT_ROOT / "data_processed"

features = pd.read_csv(PROCESSED_ROOT / "features_pre_ses1_with_posterior.csv")
features.head()

,participant_id,age,condition,n_occipital_channels,n_posterior_channels,posterior_channels,n_epochs_kept,alpha_power_8_12,posterior_alpha_sum_8_12,fooof_alpha_cf,fooof_alpha_pw,fooof_alpha_bw,fooof_r2,fooof_error
0,sub-001,60,EyesClosed,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",77,-10.257854,-9.067168,10.109051,1.711548,1.850375,0.958282,0.085840
1,sub-001,60,EyesOpen,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",35,-11.266725,-9.949122,10.994871,0.382865,3.216583,0.973884,0.050167
2,sub-002,67,EyesClosed,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",81,-10.221930,-9.088553,9.204659,1.745596,2.435050,0.966918,0.075781
3,sub-002,67,EyesOpen,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",47,-11.493045,-10.101173,8.467325,0.338656,1.327376,0.977051,0.043199
4,sub-003,44,EyesClosed,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",107,-10.989451,-9.893156,11.352306,1.386752,1.756418,0.967573,0.067330


In [11]:
features.columns.tolist()

['participant_id',
 'age',
 'condition',
 'n_occipital_channels',
 'n_posterior_channels',
 'posterior_channels',
 'n_epochs_kept',
 'alpha_power_8_12',
 'posterior_alpha_sum_8_12',
 'fooof_alpha_cf',
 'fooof_alpha_pw',
 'fooof_alpha_bw',
 'fooof_r2',
 'fooof_error']

In [12]:
df_post = features.dropna(
    subset=["participant_id", "condition", "posterior_alpha_sum_8_12"]
).copy()

X = df_post[["posterior_alpha_sum_8_12"]].values
y = (df_post["condition"] == "EyesClosed").astype(int).values
groups = df_post["participant_id"].values

print("Rows:", len(df_post))
print("Participants:", df_post["participant_id"].nunique())
df_post["condition"].value_counts()

Rows: 1195
Participants: 605


condition
EyesClosed    604
EyesOpen      591
Name: count, dtype: int64

In [13]:
cv = GroupKFold(n_splits=5)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])

scores = cross_validate(
    pipe,
    X,
    y,
    groups=groups,
    cv=cv,
    scoring=["accuracy", "roc_auc"],
    return_train_score=False
)

print(f"Posterior-alpha-only logistic regression")
print(f"Accuracy: {scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}")
print(f"ROC AUC : {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}")

Posterior-alpha-only logistic regression
Accuracy: 0.715 ± 0.013
ROC AUC : 0.786 ± 0.019


In [15]:
simple_rule_accuracy = 0.958

In [16]:
trained_accuracy = scores["test_accuracy"].mean()
trained_auc = scores["test_roc_auc"].mean()

comparison = pd.DataFrame([
    {
        "method": "Simple rule-based posterior alpha",
        "accuracy": simple_rule_accuracy,
        "auc": np.nan
    },
    {
        "method": "Logistic regression with one posterior-alpha feature",
        "accuracy": trained_accuracy,
        "auc": trained_auc
    }
])

comparison

,method,accuracy,auc
0,Simple rule-based posterior alpha,0.958000,NaN
1,Logistic regression with one posterior-alpha f...,0.714644,0.785886


In [8]:
errors = wide[~wide["correct"]].copy()
errors.head(10)

condition,participant_id,EyesClosed,EyesOpen,predicted_ec_has_higher_alpha,correct
8,sub-009,-9.439947,-9.325384,False,False
17,sub-020,-10.560258,-10.437866,False,False
66,sub-069,-10.551443,-10.455399,False,False
76,sub-079,NaN,-10.141717,False,False
105,sub-108,-9.191125,NaN,False,False
112,sub-115,-9.624584,NaN,False,False
117,sub-120,-10.555146,-10.509530,False,False
147,sub-150,-10.680086,-10.633453,False,False
155,sub-158,-9.598867,-9.536341,False,False
164,sub-167,-9.620477,NaN,False,False


In [9]:
wide["alpha_diff_ec_minus_eo"] = wide["EyesClosed"] - wide["EyesOpen"]

print(wide["alpha_diff_ec_minus_eo"].describe())

count    590.000000
mean       0.579733
std        0.370292
min       -0.236221
25%        0.271970
50%        0.565767
75%        0.836606
max        1.680422
Name: alpha_diff_ec_minus_eo, dtype: float64
